# train

In [4]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import partial_dependence
from libpysal.weights import DistanceBand, lag_spatial
from sklearn.model_selection import train_test_split
# 设置文件夹、变量
grid_folder = r'E:\seoul\480_based'
output_fig_dir = r'E:\seoul\480_based\Machine Learning\figures\nonlinear_predictions_PDP_train'
os.makedirs(output_fig_dir, exist_ok=True)

explanatory_vars_gbdt = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M','X','Y'] # 顺序很讲究
explanatory_vars = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
explanatory_vars_clean = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR']

years = [2016]

for year in years:
    # 清理后的网格数据
    file  = fr'{grid_folder}\city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'
    gdf_clean = gpd.read_file(file).replace([np.inf, -np.inf], np.nan)

    # 读取最佳参数
    best_params_file_gbdt = rf'{grid_folder}\Machine Learning\final_GBDT_summary_results.xlsx'
    best_df_gbdt = pd.read_excel(best_params_file_gbdt)
    best_params_file_sdem = rf'{grid_folder}\statistics\SDEM_all_params.xlsx'
    best_df_sdem = pd.read_excel(best_params_file_sdem)

    # 输出文件夹
    output_dir = os.path.join(output_fig_dir, f'{year}')
    os.makedirs(output_dir, exist_ok=True)
    target_folder = best_df_gbdt['Target'].unique()
    print(target_folder)

    for target in target_folder[1:]:

        # 先收集 EXT 和 NOR 的预测数据
        ext_targets = [t for t in target_folder if ('ext' in t.lower()) or ('nor' in t.lower())]
        hr_targets = [t for t in target_folder if 'hr' in t.lower()]

        # ---------------- EXT + NOR 合并 ----------------
        for feature in explanatory_vars:
            plt.figure(figsize=(8,5))
            for target in ext_targets:
                # gbdt
                row_gbdt = best_df_gbdt[best_df_gbdt['Target']==target].iloc[0]
                params_gbdt = {
                    'learning_rate': row_gbdt['learning_rate'],
                    'max_depth': int(row_gbdt['max_depth']),
                    'n_estimators': int(row_gbdt['n_estimators']),
                    'subsample': row_gbdt['subsample'],
                    'min_samples_split': int(row_gbdt['min_samples_split']),
                    'max_features': float(row_gbdt['max_features']),
                    'random_state': 0
                }

                np.random.seed(0)  # 固定种子以便复现
                random_seeds = np.random.choice(10000, size=20, replace=False)

                X = gdf_clean[explanatory_vars_gbdt]
                y = gdf_clean[target]
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = row_gbdt["Random seed"])
                #X_grid = np.linspace(X_test[feature].min(), X_test[feature].max(), 100)
                # X_base = pd.DataFrame(np.tile(X.mean().values, (100,1)), columns=explanatory_vars_gbdt)
                # X_base[feature] = X_grid
                # y_gbdt_pred = GradientBoostingRegressor(**params_gbdt).fit(X, gdf_clean[target]).predict(X_base)
                model = GradientBoostingRegressor(**params_gbdt).fit(X_train, y_train)
                # ----------- PDP 计算部分 -----------
                pdp_result = partial_dependence(
                    model,
                    X_train,
                    [feature],
                    grid_resolution=100,
                    kind='average'
                )

                X_grid = pdp_result['values'][0]
                y_pdp_pred = pdp_result['average'][0] + y_train.mean()
                print(y_pdp_pred)

                # SDEM
                row_sdem = best_df_sdem[best_df_sdem['Target']==target].iloc[0]
                params_sdem = {
                    'CONSTANT': row_sdem['CONSTANT'],
                    'BCR': row_sdem['BCR'],
                    'BHV': row_sdem['BHV'],
                    'SVF': row_sdem['SVF'],
                    'NDVI': row_sdem['NDVI'],
                    'EV': row_sdem['EV'],
                    'WR': row_sdem['WR'],
                    'Dist_W': row_sdem['Dist_W'],
                    'Dist_P': row_sdem['Dist_P'],
                    'Dist_M': row_sdem['Dist_M'],
                    'W_BCR': row_sdem['W_BCR'],
                    'W_BHV': row_sdem['W_BHV'],
                    'W_SVF': row_sdem['W_SVF'],
                    'W_NDVI': row_sdem['W_NDVI'],
                    'W_EV': row_sdem['W_EV'],
                    'W_WR': row_sdem['W_WR'],
                    'lambda': row_sdem['lambda']
                }
                # SDEM 预测
                X_mean = gdf_clean[explanatory_vars].mean().to_dict()
                w = DistanceBand.from_dataframe(gdf_clean, threshold=1000, binary=False)
                w.transform = 'r'
                WX = lag_spatial(w, gdf_clean[explanatory_vars_clean].values)
                WX_mean = pd.Series(WX.mean(axis=0), index=[f"W_{v}" for v in explanatory_vars_clean])
                y_sdem_pred = []
                for val in X_grid:
                    X_temp = X_mean.copy()
                    X_temp[feature] = val
                    WX_temp = WX_mean.copy()
                    if feature in explanatory_vars_clean:
                        WX_temp[feature] = pd.Series(val, index=[f"W_{feature}"])
                    all_vars = {'CONSTANT': 1.0, **X_temp, **WX_temp.to_dict()}
                    y_pred = 0
                    for var, coef in params_sdem.items():
                        if var != 'lambda':
                            y_pred += coef * all_vars.get(var, 0)
                    y_sdem_pred.append(y_pred)
                y_sdem_pred = np.array(y_sdem_pred)

                # 根据 EXT/NOR 指定颜色
                if 'ext' in target.lower():
                    color_gbdt = '#333333'  # 黑色
                    color_sdem = '#555555'  # 黑色虚线
                else:  # NOR
                    color_gbdt = '#0000ff'  # 蓝色
                    color_sdem = '#0000ff'  # 蓝色虚线

                # 命名规范映射
                target_name_map = {
                    'nor_2016': 'Nor_LST',
                    'ext_2016': 'Ext_LST',
                    'hr_2016': 'HR'
                }
                label_name = target_name_map.get(target, target)  # 如果没有在字典里就用原名
                # 绘图
                plt.plot(X_grid, y_pdp_pred, color=color_gbdt, linewidth=2, label=f'{label_name} GBDT')
                plt.plot(X_grid, y_sdem_pred, color=color_sdem, linestyle='--', linewidth=2, label=f'{label_name} SDEM')

            plt.xlabel(feature)
            plt.ylabel('Temperature (%)')
            plt.grid(True)
            plt.legend()
            plt.ylim(20,55)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f'{feature}_ext_nor_{year}_comparison.png'), dpi=300)
            plt.close()
            print(f"✅ Saved {feature}_ext_nor_{year}_comparison.png")

['nor_2016' 'ext_2016' 'hr_2016']


E:\python\lib\site-packages\sklearn\utils\_bunch.py:35: FutureWarning: Key: 'values', is deprecated in 1.3 and will be removed in 1.5. Please use 'grid_values' instead.
  warnings.warn(
E:\python\lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)


[29.97614757 30.20454322 30.12383313 30.06588533 30.05813757 30.07652141
 30.03155511 30.05929141 30.23767521 30.2250692  30.23469935 30.23965221
 30.29887732 30.29683678 30.50929131 30.7473854  30.82700831 30.81730212
 30.77963188 30.82299985 30.81647825 30.86780339 30.97887041 30.98015533
 30.89092063 30.87320383 30.87231394 31.0394527  31.1348644  31.1660745
 31.16950301 31.19328303 31.18358715 31.18760925 31.17937749 31.30552916
 31.44503129 31.45029142 31.53686439 31.57214877 31.47245655 31.45065762
 31.41886703 31.42889769 31.44483167 31.43370339 31.54969115 31.62319201
 31.6594396  31.65834355 31.63311092 31.64039507 31.64648616 31.6652832
 31.63690734 31.63217534 31.60897748 31.64735271 31.73034463 31.76920033
 31.72194928 31.74830958 31.71816241 31.70630111 31.7296646  31.75003987
 31.76781413 31.76427461 31.74179591 31.76044081 31.75882895 31.80011901
 31.78391837 31.80370122 31.76984014 31.77209364 31.81129927 31.88269059
 31.87376923 31.88565759 31.8776513  31.90486    31.9

E:\python\lib\site-packages\libpysal\weights\weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  warnings.warn(message)


('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')


...

[30.87178141 31.01438006 31.09195779 31.09147403 31.03338528 31.11285071
 31.11285551 31.12208438 31.09300372 31.06812729 31.08653705 31.07780517
 31.05500151 31.07729084 31.06692642 31.0890197  31.0711338  31.08985599
 31.07794622 31.07167007 31.0673458  31.07586704 31.08202356 31.09387733
 31.11390647 31.10265505 31.08819662 31.0582931  31.11212946 31.0916997
 31.0680251  31.08052844 31.08815985 31.07672175 31.11344507 31.10341832
 31.06308884 31.06819861 31.04702325 31.04465097 31.0600607  31.04976512
 31.03540131 31.0222565  30.99765783 31.00153368 30.97999477 30.97566778
 30.98526687 30.98125057 30.97331559 30.98248451 30.98349721 30.97555644
 30.94392081 30.92128342 30.91709194 30.91308967 30.92981583 30.92624469
 30.9263875  30.9281543  30.91571549 30.92254112 30.95829066 30.96525958
 30.98322801 31.05100021 31.07117423 31.05309813 30.98456184 30.9987581
 31.00264368 31.02313467 31.02059221 31.0530174  31.0530897  31.04852632
 31.04496359 31.04730961 31.04378909 31.04658866 31.0

E:\python\lib\site-packages\sklearn\utils\_bunch.py:35: FutureWarning: Key: 'values', is deprecated in 1.3 and will be removed in 1.5. Please use 'grid_values' instead.
  warnings.warn(


[41.28238937 41.46479342 41.51389627 41.55976163 41.59772564 41.61948506
 41.63495895 41.63422298 41.62922082 41.63383417 41.62901084 41.58888074
 41.57337104 41.58022354 41.55810355 41.55479775 41.54986527 41.56922811
 41.55345492 41.56621634 41.57030948 41.58589272 41.57784266 41.59387316
 41.60117687 41.6056817  41.59558293 41.59183104 41.66888048 41.65684334
 41.60054373 41.59162558 41.55578886 41.48226249 41.70059885 41.6062853
 41.59232125 41.57557011 41.58800784 41.58258054 41.58065813 41.56359706
 41.50893311 41.56185354 41.55166402 41.54480011 41.54675098 41.52875101
 41.54820774 41.51335752 41.50083895 41.44663295 41.45133255 41.44097523
 41.43258318 41.33719233 41.26204975 41.25587388 41.29353069 41.27889571
 41.282476   41.29116656 41.30308434 41.32240554 41.34223369 41.32195728
 41.33475111 41.41244178 41.40678092 41.37403908 41.37390278 41.38622973
 41.39907087 41.42539612 41.47089706 41.48895987 41.46796099 41.47621435
 41.48784168 41.49540266 41.44006487 41.46570091 41.

E:\python\lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
E:\python\lib\site-packages\libpysal\weights\weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  warnings.warn(message)


('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')
✅ Saved Dist_M_ext_nor_2016_comparison.png
